# Lecke 11 - Ügynök-ügynök (A2A) protokoll


## Beállítás


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Mi az A2A protokoll?

A **Agent-to-Agent (A2A) protokoll** egy nyílt szabvány, amely lehetővé teszi, hogy a mesterséges intelligencia ügynökök kommunikáljanak,
felfedezzék egymást, és együttműködjenek — még akkor is, ha különböző keretrendszereken épülnek vagy
különböző szolgáltatásoknál vannak hosztolva.

Kulcsfogalmak:

- **Felfedezés** – Az ügynökök közzétesznek egy *Ügynök-kártyát*, amely leírja a képességeiket, így
  könnyű a többi ügynök (vagy az összehangoló) számára megtalálni a megfelelő szakértőt egy feladathoz.
- **Üzenetküldés** – Az ügynökök strukturált üzeneteket cserélnek egy közös protokollon keresztül, így egy
  kérés egy ügynöktől megérthető és teljesíthető egy másik által, függetlenül annak belső
  megvalósításától.
- **Feladat életciklusa** – Az A2A olyan állapotokat határoz meg, mint a *submitted*, *working*, *completed*, és
  *failed*, teljes rálátást biztosítva az összehangolónak arra, hogyan halad egy delegált feladat.

Ebben a leckében az A2A-stílusú együttműködést szimuláljuk azzal, hogy három specializált utazási ügynököt
bekötünk egy munkafolyamatba, ahol minden ügynök hozzájárul a szakértelmével és továbbadja az eredményeket a következőnek.


## Specializált utazási ügynökök létrehozása


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Többügynökös együttműködés munkafolyamattal

A három ügynököt összekapcsoljuk egy sorozatos munkafolyamatba, amely tükrözi az A2A üzenetküldést:

1. **CurrencyExchangeAgent** átveszi a felhasználó kérését és valutaváltási útmutatást ad.
2. **ActivityPlannerAgent** átveszi a kibővített kontextust és tevékenységajánlásokat ad hozzá.
3. **TravelManagerAgent** mindkét bemenetet egyesíti egy végső utazási összefoglalóvá.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## A2A megértése éles környezetben

Egy éles környezetben az A2A protokoll hatékony szolgáltatások közötti forgatókönyveket tesz lehetővé:

| Capability | Description |
|---|---|
| **Cross-framework interop** | Egy adott keretrendszerrel készült ügynök feladatokat adhat át egy másik, A2A-kompatibilis keretrendszerrel készült ügynöknek, lehetővé téve az igazi szervezetek közötti interoperabilitást. |
| **Service boundaries** | Az ügynökök külön mikroservizekben, felhő régiókban, vagy akár eltérő szervezeteknél is futhatnak, miközben zökkenőmentesen együttműködnek. |
| **Dynamic discovery** | Egy orchestrator futásidőben lekérdezhet egy Agent Card regisztert, hogy megtalálja az adott alfeladathoz leginkább megfelelő specialistát. |
| **Streaming & push notifications** | Az A2A támogatja a Server-Sent Events (SSE) valós idejű előrehaladás-frissítésekhez és push értesítéseket a hosszú ideig futó feladatokhoz. |

A fenti munkafolyamat, amelyet felépítettünk, ennek a mintának egy egyszerűsített, ugyanazon folyamaton belüli változata. Egy valódi telepítésnél minden ügynök HTTP végpontot nyitna, publikálná az Agent Cardot, és az A2A JSON-RPC protokollon keresztül kommunikálna.


## Összegzés

Ebben a leckében megtanultad:

1. **Mi az A2A protokoll** — egy nyílt szabvány ügynökök közötti felfedezésre, üzenetküldésre,
   és feladatkezelésre.
2. **Hogyan hozz létre speciális ügynököket** — egy Valutaátváltó ügynököt, egy Tevékenységtervező ügynököt és egy Utazáskezelő koordinátort.
3. **Hogyan kötöd össze az ügynököket egy munkafolyamatba** — `WorkflowBuilder` használatával az ügynökök közötti sorozatos üzenetküldés modellezéséhez.
4. **Hogyan működik az A2A éles környezetben** — lehetővé téve a keretrendszerek és szolgáltatások közötti együttműködést dinamikus felfedezéssel és folyamatos frissítésekkel.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Felelősségkizárás:
Ezt a dokumentumot az AI fordítószolgáltatás, a Co-op Translator (https://github.com/Azure/co-op-translator) segítségével fordítottuk. Bár igyekszünk pontos fordítást biztosítani, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum az anyanyelvén tekintendő a hiteles forrásnak. Kritikus jelentőségű információk esetén szakmai, emberi fordítást javaslunk. Nem vállalunk felelősséget az ebből a fordításból eredő félreértésekért vagy helytelen értelmezésekért.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
